In [ ]:
%reset -f

### Initialize MCL Z positioner

In [ ]:
from hardware.mclController import MCLStage, loadMCLDLL, ProductInformation
import time

print("Initializing Stage")
try: # if stage was already initialized, shut it down first
    stage.shutDown()
    time.sleep(0.5)
except NameError:
    pass

# DLL file location
mcl_lib = "c:/Program Files/Mad City Labs/NanoDrive/Madlib"  

stage = MCLStage(mcl_lib)
if not stage.getStatus():
    exit()

print("Stage Properties:")
stageProperties = stage.getProperties()
for key in sorted(stageProperties):
        print(key, '\t', stageProperties[key])
print("")

print("Z axis range:", "%.2f um" % stage.getAxisRange(3))
print("")

zPosition = 50 # initial z position in microns
stage.zMoveTo(zPosition)
time.sleep(0.05) # let it move, minimum wait for 0.05s

stage.getPosition(3)
print("Current Z Position:", "%.4f um" %stage.getPosition(3))
print("")

In [ ]:
import hardware.SPAD512S as SPAD512S
import pySPADutils as util

port = 9990 # Check the command Server in the setting tab of the software and change it if necessary
SPAD1 = SPAD512S(port)

info = SPAD1.get_info()
print("\nGeneral informations of the camera :")
print(info)
temp = SPAD1.get_temps() # Current temperatures of FPGAs, PCB and Chip
print("\nCurrent temperatures of FPGAs, PCB and Chip :")
print(temp)
freq = SPAD1.get_freq() # Operating frequencies (Laser and frame)
print("\nOperating frequencies (Laser and frame) :")
print(freq)
SPAD1.enable_cooling(1) # Enable the cooling system of the camera

In [ ]:
# Camera settings
overlap = 1
timeout = 0
pileup = 1

im_width = 512
rowCol = 512 #Number of rows/columns
bitdepth = 1

# Each layer
BP = 50000
intTime = 10 # in us if 1-bit

In [ ]:
# Z scan settings
# Z mode 0: single layer, 1: Z stack
Z_mode = 0 

Z_stepSize = 0.1    # in microns
Z_range = 10        # in microns

Z_middle = 50 
n_steps = int(Z_range/Z_stepSize) + 1
Z_positions = [Z_middle + (i-n_steps//2)*Z_stepSize for i in range(n_steps)]

print("Z positions to scan (in microns):", Z_positions)

In [ ]:
import pySPADutils as util
from pytictoc import TicToc

#TODO: implement Z scan loop, allocation

t = TicToc()
if Z_mode == 0:
    print("Acquiring single layer")
    t.tic()
    data = SPAD1.get_intensity_1bit_packed(BP, intTime, bitdepth, overlap, timeout, pileup, im_width)
    t.toc("Acquisition time took")
else:
    data = []
    for z in Z_positions:
        stage.zMoveTo(z)
        time.sleep(0.05) # let it move, minimum wait for 0.05s
        t.tic()
        data_z = SPAD1.get_intensity_1bit_packed(BP, intTime, bitdepth, overlap, timeout, pileup, im_width)
        t.toc(f"Acquisition time at Z={z:.2f} um took")
        data.append(data_z)

In [ ]:
filePath = "test.bin"
util.writeBinBig(filePath, data)

In [ ]:
stage.shutDown()